## From Building Blocks to U-Net - Understanding Architectural Innovation

### Part 1: Understanding U-Net Architecture (15 points)

U-Net was introduced by Ronneberger et al. (2015) for biomedical image segmentation. The architecture won the ISBI cell tracking challenge 2015 by a large margin.

![U-Net Architecture](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png)
*Figure 1: Original U-Net architecture (Ronneberger et al., 2015)*

### Starter Code: Basic U-Net Implementation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

**Q1.1 (5 points):** Identify and explain the three major architectural components/concepts from our previous lectures that U-Net combines. For each component:
- Name the concept
- Explain how it's implemented in U-Net
- Describe what problem it solves

**Name of Concept 1**: 3x3 Convolutional blocks 

**Description of how Concept 1 is implemented in U-Net**: 
This is implemented via the DoubleConv() class 

**Problem Concept 1 solves**: 
These allow the model to learn detailed features of images without using a massive amount of parameters. 

-----

**Name of Concept 2**: Encoder/Decoder

**Description of how Concept 2 is implemented in U-Net**: 
These are implemented via the classes Down() (encoder) and Up() (decoder). 

**Problem Concept 2 solves**: 
This allows the UNet system to comprehend the whole image rather than just a pixle at a time. 

-----

**Name of Concept 3**: Skip Connection

**Description of how Concept 3 is implemented in U-Net**: 
This is implemented as part of the decoder class (Up()) as x = torch.cat([x2, x1], dim=1)

**Problem Concept 3 solves**: 
This helps overcome some of the edge detail loss that is experienced as part of the encoder/decoder bottleneck. 

**Q1.2 (5 points):** Skip connections in U-Net use **concatenation** rather than **addition** (as in ResNet).
- What are the implications of this design choice?
- How does this affect the number of parameters?
- When might addition be preferred over concatenation?

**Implications of having skip connections in U-Net use concatenation**: 
Using concatenation allows both input signals to stay separate from each other and intact. Adding them would cause the two signals to get mixed together. It also helps with mem

**Effect of concatenation on number of parameters**: 
Concatenation increases the number of input channels to the convolution step that comes after. This about doubles the weights in the first convolution of each decoder block. 

**When addition might be preferred over concatenation**: 
Addition may be better when the two signals are very similar, and you dont gain much from keeping them separate. There is also a memory consideration, as the concatenation approach results in a much larger tensor for the backprop process to handle. 

**Q1.3 (5 points):** Critical Analysis:
- What are the potential limitations of the original U-Net architecture?
- How might U-Net struggle with very high-resolution images?
- Propose at least two modifications that could address these limitations.

**Potential limitations of the original U-Net architecture**: 
The original version uses a fixed number of down sampling levels and a fixed set of channel widths. This causes it to poorly adapt to images of varying size that may require different depths in order to understand correctly. 

**How U-Net might struggle with very high-resolution images**: 
Since the architecture fixes the receptive field, a larger image will cause the each feature to only see a small subsection of the image. Higher resolution images may aslo cause further memory challenges as the larger pixel count will require the feature maps to scale up much faster as well. 

**Proposed modifications**: 
To address the fixed down sampling and channel width issue, the model could potentially be made more dynamic and able to alter its structure conditionally based on the image it is dealing with. To address the memory issues I have mentioned, the model could perhaps use either concatenation or addition-oriented skip connections based on the size of the image and the known computational limitations at the time of application. 

### Part 2: Architectural Variants (15 points)

#### Variant 1: U-Net++ (Nested U-Net)

U-Net++ introduces nested skip connections to reduce the semantic gap between encoder and decoder features.

In [ ]:
class UNetPlusPlus(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Define the nested structure
        # X^{0,0} -> X^{1,0} -> X^{2,0} -> X^{3,0} -> X^{4,0}
        #    |         |         |         |
        # X^{0,1} -> X^{1,1} -> X^{2,1} -> X^{3,1}
        #    |         |         |
        # X^{0,2} -> X^{1,2} -> X^{2,2}
        #    |         |
        # X^{0,3} -> X^{1,3}
        #    |
        # X^{0,4}

        nb_filter = [32, 64, 128, 256, 512]

        # Nested skip connections would be implemented here.
        # This is a simplified structure for illustration.
        pass

#### Variant 2: V-Net (Volumetric Convolutional Neural Networks)

V-Net extends U-Net to 3D volumetric data and adds residual connections within each stage.

In [ ]:
class VNetBlock(nn.Module):

    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv3d(in_channels if i == 0 else out_channels,
                     out_channels, kernel_size=3, padding=1)
            for i in range(num_convs)
        ])
        self.activation = nn.PReLU(out_channels)

        self.residual = nn.Conv3d(in_channels, out_channels, kernel_size=1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        out = x
        for conv in self.convs:
            out = self.activation(conv(out))

        return out + residual

**Q3.1 (5 points):** U-Net++ Analysis:
- What problem does the nested skip connection architecture solve?
- How does it reduce the "semantic gap"?
- What is the computational cost compared to standard U-Net?

**Problem nested skip connection architecture solves**: 
As images are evaluated and the convolution process is applied several times there is a loss of meaning between the different steps. This delta of understood meaning between the layers is the semantic gap. The nested skip connections in U-Net++ aim to address this. 

**How nested skip connection architecture reduces semantic gap**: 
The way I think of this is that at each nested step there is a 'mico-learning' that occurs. Rather than having the semantic gap grow at each step in the process, the nested skip connections allow the semantic gap to be reduced as the meaning is reiterated several times in the process. 

**Computational cost of U-Net with nested skip connection architecture**: 
While the number of parameters do not drastically increase from the addition of nest skip connections, the amount of memory required goes up by more that double, and the amount of comutational power scales up by a factor of 3. 

In [ ]:

class VNetBlock(nn.Module):

    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv3d(in_channels if i == 0 else out_channels,
                     out_channels, kernel_size=3, padding=1)
            for i in range(num_convs)
        ])
        self.activation = nn.PReLU(out_channels)

        self.residual = nn.Conv3d(in_channels, out_channels, kernel_size=1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        out = x
        for conv in self.convs:
            out = self.activation(conv(out))

        return out + residual

**Q3.2 (5 points):** V-Net Contributions:
- Besides 3D extension, what are the key innovations in V-Net?
- Why are residual connections particularly important for volumetric data?
- How does the Dice loss implementation differ for 3D data?

**Key innovations in V-Net**: 
A few of the key innovations are the change from pooling operations to convolutional operations that can have much smaller memory footprint during training in many situations. The V-Net also uses a learned residual function rather than a predefined one. The V-Net architecture ends up using fewer channels than U-Net, which helps offset the cost of the larger kernel needed for 3D data.  

**Why residual connections are important for volumetric data**: 
The higher memory of 3D data requires everything to be more efficient in its use of weights, and the residual connections help in this regard. Also volumetric data can be expensive to acquire, and the use of residual connections can make a great deal of progress with a relatively small dataset. 

**How Dice loss differs for 3D data**: 
In situations where the area of interest tends to be a small region, the use of typical loss functions are proned to being trapped in local minima and can cause the models to be biased towards the background of the images. The Dice loss aims to maximize the Dice coefficient which exists between 0 and 1. This coefficient optimization has the effect of refocusing the model on the foreground as well as the background without using sample re-weighting. 

**Q3.3 (5 points):** Comparative Analysis:
Create a comparison table with at least 5 criteria comparing:
- Standard U-Net
- U-Net++
- V-Net

Consider: parameter count, memory usage, suitable applications, training difficulty, and inference speed.

| Criterion | Standard U-Net | U-Net++ | V-Net |
| ----- | ----- | ----- | ----- |
| Parameter Count | 7.8M - 31M | 9M - 36M | 65M - 70M |
| Memory Usage | Low | Low-Moderate | High |
| Suitable Applications | 2D images with fixed size | Images that require boundary precision | Volumetric data and 3D images |
| Training Difficulty | Easy | Moderate; more hyperparameters | Hardest; small batch sizes; long training |
| Inference Speed | Fastest | Slower | Slowest |

## Resources

### Papers
- [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
- [UNet++: A Nested U-Net Architecture](https://arxiv.org/abs/1807.10165)
- [V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation](https://arxiv.org/abs/1606.04797)

### Implementations
- [PyTorch U-Net](https://github.com/milesial/Pytorch-UNet)
- [Segmentation Models PyTorch](https://github.com/qubvel/segmentation_models.pytorch)
- [MONAI (Medical Imaging)](https://monai.io/)